# Huấn luyện mô hình Semantic Segmentation — CARLA

**Đồ án:** Nghiên cứu ứng dụng học tăng cường sâu (DRL) điều khiển xe bám làn mô phỏng trong môi trường CARLA.

**Vai trò của notebook này trong đồ án:** đây là bước 1/3 của pipeline. Model segmentation
sinh ra biểu diễn trạng thái (state) gọn — ảnh gray class-ID — thay vì dùng RGB thô, giúp
giảm chiều dữ liệu đầu vào và loại bỏ nhiễu (màu sắc, ánh sáng, texture) không liên quan
trực tiếp đến việc bám làn. Model này sẽ được **đóng băng** khi dùng cho bước IL (notebook 2)
và fine-tune DRL (bước 3, chưa thực hiện trong notebook này).

**Nội dung notebook:**
1. Cài đặt & import
2. Cấu hình + Dataset
3. Trực quan hóa dữ liệu (ảnh/mask mẫu, phân bố class)
4. Định nghĩa model (DeepLabV3+ / resnet34)
5. Huấn luyện + theo dõi mIoU
6. Trực quan hóa kết quả dự đoán trên tập validation


## 1. Cài đặt thư viện

Bỏ comment dòng dưới nếu chạy trên Google Colab / máy chưa cài.

In [ ]:
# !pip install segmentation-models-pytorch albumentations opencv-python-headless torch torchvision tqdm matplotlib


In [ ]:
import os
import random
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Cấu hình & Reproducibility

**Chỉnh lại đường dẫn dataset (`TRAIN_IMG_DIR`, ...) cho khớp máy bạn.**

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

IMAGE_HEIGHT = 384
IMAGE_WIDTH = 480
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 2
USE_AMP = torch.cuda.is_available()
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 6
CLASS_WEIGHT_SAMPLE = 200

NUM_CLASSES = 13  # theo bảng màu CARLA bạn dùng để gán nhãn (class ID 0..12)

TRAIN_IMG_DIR = "./carla_dataset/train_images"
TRAIN_MASK_DIR = "./carla_dataset/train_masks"
VAL_IMG_DIR = "./carla_dataset/val_images"
VAL_MASK_DIR = "./carla_dataset/val_masks"

CHECKPOINT_PATH = "best_carla_segmentation.pth"

# Palette hiển thị — chỉnh lại nếu bạn có bảng màu class gốc riêng
PALETTE = np.array([
    [0, 0, 0], [70, 70, 70], [190, 153, 153], [250, 170, 160],
    [220, 20, 60], [153, 153, 153], [157, 234, 50], [128, 64, 128],
    [244, 35, 232], [107, 142, 35], [0, 0, 142], [102, 102, 156],
    [220, 220, 0],
], dtype=np.uint8)


## 3. Dataset

Khớp cặp ảnh–mask theo **tên file** (không dựa vào thứ tự `sorted()` đơn thuần) để tránh
lệch nhãn âm thầm nếu thiếu file. Mask CARLA lưu class ID ở kênh Đỏ (R) — do OpenCV đọc
theo thứ tự BGR nên R nằm ở index 2.

In [ ]:
class CarlaSegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.transform = transform
        img_files = {os.path.splitext(f)[0]: f for f in os.listdir(images_dir)}
        mask_files = {os.path.splitext(f)[0]: f for f in os.listdir(masks_dir)}
        common_keys = sorted(set(img_files.keys()) & set(mask_files.keys()))
        if not common_keys:
            raise RuntimeError(f"Không tìm thấy cặp ảnh-mask hợp lệ trong {images_dir} / {masks_dir}")
        self.images = [os.path.join(images_dir, img_files[k]) for k in common_keys]
        self.masks = [os.path.join(masks_dir, mask_files[k]) for k in common_keys]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = cv2.imread(self.images[idx], cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask_raw = cv2.imread(self.masks[idx], cv2.IMREAD_UNCHANGED)
        mask = mask_raw[:, :, 2] if mask_raw.ndim == 3 else mask_raw

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented["image"], augmented["mask"]
        return image, mask.long()


_resize_kwargs = dict(height=IMAGE_HEIGHT, width=IMAGE_WIDTH, interpolation=cv2.INTER_LINEAR)
try:
    _resize = A.Resize(**_resize_kwargs, mask_interpolation=cv2.INTER_NEAREST)
except TypeError:
    print("[WARN] albumentations bản cũ không hỗ trợ mask_interpolation riêng -> nâng cấp để tránh nội suy sai nhãn.")
    _resize = A.Resize(**_resize_kwargs)

train_transform = A.Compose([
    _resize,
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
val_transform = A.Compose([
    _resize,
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_ds = CarlaSegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
val_ds = CarlaSegmentationDataset(VAL_IMG_DIR, VAL_MASK_DIR, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} mẫu | Val: {len(val_ds)} mẫu")


## 4. Trực quan hóa dữ liệu mẫu

Kiểm tra nhanh ảnh + mask (tô màu theo `PALETTE`) đã khớp đúng chưa trước khi train.

In [ ]:
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.permute(1, 2, 0).numpy()
    img = (img * std + mean).clip(0, 1)
    return img

fig, axes = plt.subplots(3, 2, figsize=(10, 12))
for i in range(3):
    img, mask = train_ds[random.randint(0, len(train_ds) - 1)]
    axes[i, 0].imshow(denormalize(img))
    axes[i, 0].set_title("Ảnh RGB")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(PALETTE[mask.numpy().clip(0, NUM_CLASSES - 1)])
    axes[i, 1].set_title("Mask (tô màu)")
    axes[i, 1].axis("off")
plt.tight_layout()
plt.show()


## 5. Class weights — xử lý lệch lớp

Vạch kẻ đường/vỉa hè... chiếm rất ít pixel so với mặt đường/bầu trời. Dùng median-frequency balancing.

In [ ]:
def compute_class_weights(mask_paths, num_classes, sample_size=None):
    paths = random.sample(mask_paths, sample_size) if sample_size and sample_size < len(mask_paths) else mask_paths
    counts = np.zeros(num_classes, dtype=np.int64)
    for p in tqdm(paths, desc="Đang quét mask để tính class weights"):
        m = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if m is None:
            continue
        m = m[:, :, 2] if m.ndim == 3 else m
        vals, cnts = np.unique(m, return_counts=True)
        for v, c in zip(vals, cnts):
            if 0 <= v < num_classes:
                counts[v] += c
    counts = np.maximum(counts, 1)
    freq = counts / counts.sum()
    weights = np.median(freq) / freq
    return torch.tensor(weights, dtype=torch.float32), counts

class_weights, class_pixel_counts = compute_class_weights(train_ds.masks, NUM_CLASSES, sample_size=CLASS_WEIGHT_SAMPLE)
class_weights = class_weights.to(DEVICE)

plt.figure(figsize=(10, 4))
plt.bar(range(NUM_CLASSES), class_pixel_counts, color=[tuple(c/255 for c in PALETTE[i]) for i in range(NUM_CLASSES)])
plt.yscale("log")
plt.xlabel("Class ID"); plt.ylabel("Số pixel (log scale)")
plt.title("Phân bố pixel theo class trong tập mẫu")
plt.xticks(range(NUM_CLASSES))
plt.show()

print("Class weights:", class_weights.cpu().numpy().round(3))


## 6. Model, Loss, Optimizer

In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(DEVICE)

criterion_ce = nn.CrossEntropyLoss(weight=class_weights)
criterion_dice = smp.losses.DiceLoss(mode="multiclass")

def combined_loss(outputs, masks):
    return criterion_ce(outputs, masks) + criterion_dice(outputs, masks)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Số tham số model: {n_params:.2f}M")


## 7. Vòng lặp huấn luyện

mIoU được tích lũy trên **toàn bộ** tập validation (không lấy trung bình theo batch) để tránh sai lệch.

In [ ]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    tp_sum = fp_sum = fn_sum = tn_sum = None

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, masks in loader:
            images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(images)
                loss = combined_loss(outputs, masks)

            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            tp, fp, fn, tn = smp.metrics.get_stats(preds, masks, mode="multiclass", num_classes=NUM_CLASSES)
            if tp_sum is None:
                tp_sum, fp_sum, fn_sum, tn_sum = tp, fp, fn, tn
            else:
                tp_sum += tp; fp_sum += fp; fn_sum += fn; tn_sum += tn

    miou = smp.metrics.iou_score(tp_sum, fp_sum, fn_sum, tn_sum, reduction="macro").item()
    return total_loss / len(loader), miou


In [ ]:
history = {"train_loss": [], "val_loss": [], "val_miou": [], "epoch_time": [], "lr": []}
best_miou = 0.0
epochs_no_improve = 0
training_start = time.time()

pbar = tqdm(range(EPOCHS), desc="Training")
for epoch in pbar:
    t0 = time.time()
    train_loss, _ = run_epoch(train_loader, train=True)
    val_loss, val_miou = run_epoch(val_loader, train=False)
    scheduler.step()
    epoch_time = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(val_miou)
    history["epoch_time"].append(epoch_time)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    pbar.set_postfix({"train_loss": f"{train_loss:.4f}", "val_loss": f"{val_loss:.4f}", "val_mIoU": f"{val_miou:.4f}"})

    if val_miou > best_miou:
        best_miou = val_miou
        epochs_no_improve = 0
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(),
            "best_miou": best_miou, "num_classes": NUM_CLASSES,
            "image_height": IMAGE_HEIGHT, "image_width": IMAGE_WIDTH,
        }, CHECKPOINT_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Dừng sớm ở epoch {epoch+1} (không cải thiện {EARLY_STOP_PATIENCE} epoch liên tiếp)")
            break

total_time_min = (time.time() - training_start) / 60
print(f"Hoàn tất. Best mIoU: {best_miou:.4f} | Tổng thời gian train: {total_time_min:.1f} phút | Checkpoint: {CHECKPOINT_PATH}")


## 8. Biểu đồ huấn luyện

Dùng để chèn vào báo cáo đồ án.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].set_title("Loss theo epoch")

axes[1].plot(history["val_miou"], color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mIoU"); axes[1].set_title("Val mIoU theo epoch")

axes[2].plot(history["lr"], color="orange")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Learning Rate"); axes[2].set_title("LR schedule (Cosine)")
plt.tight_layout()
plt.show()

print(f"Thời gian trung bình/epoch: {np.mean(history['epoch_time']):.1f}s")


## 9. Đánh giá chi tiết theo từng class — Per-class IoU & Confusion Matrix

Dùng checkpoint tốt nhất, đánh giá trên toàn bộ tập validation. 2 biểu đồ này rất nên đưa
vào báo cáo đồ án — cho thấy model yếu ở class nào (thường là các class hiếm/diện tích nhỏ
như vạch kẻ đường).

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

def compute_confusion_matrix(model, loader, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Đánh giá val"):
            images = images.to(DEVICE)
            preds = torch.argmax(model(images), dim=1).cpu().numpy().reshape(-1)
            gts = masks.numpy().reshape(-1)
            valid = (gts >= 0) & (gts < num_classes)
            idx = gts[valid] * num_classes + preds[valid]
            cm += np.bincount(idx, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    return cm

cm = compute_confusion_matrix(model, val_loader, NUM_CLASSES)

# Per-class IoU = TP / (TP + FP + FN)
tp = np.diag(cm)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp
per_class_iou = tp / np.maximum(tp + fp + fn, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(range(NUM_CLASSES), per_class_iou, color=[tuple(c/255 for c in PALETTE[i]) for i in range(NUM_CLASSES)])
axes[0].set_xlabel("Class ID"); axes[0].set_ylabel("IoU"); axes[0].set_title("Per-class IoU (tập Validation)")
axes[0].set_xticks(range(NUM_CLASSES)); axes[0].set_ylim(0, 1)
for i, v in enumerate(per_class_iou):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=8)

cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
im = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
axes[1].set_xlabel("Dự đoán"); axes[1].set_ylabel("Thực tế"); axes[1].set_title("Confusion Matrix (chuẩn hoá theo hàng)")
axes[1].set_xticks(range(NUM_CLASSES)); axes[1].set_yticks(range(NUM_CLASSES))
plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

print("Bảng tổng kết:")
print(f"{'Class ID':<10}{'IoU':<10}")
for i in range(NUM_CLASSES):
    print(f"{i:<10}{per_class_iou[i]:<10.4f}")
print(f"\nmIoU (macro): {per_class_iou.mean():.4f}")


## 10. Trực quan hóa dự đoán trên tập validation

So sánh Ảnh gốc / Ground Truth / Dự đoán — dùng checkpoint tốt nhất.

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

fig, axes = plt.subplots(3, 3, figsize=(13, 12))
with torch.no_grad():
    for i in range(3):
        img, mask = val_ds[random.randint(0, len(val_ds) - 1)]
        pred = torch.argmax(model(img.unsqueeze(0).to(DEVICE)), dim=1)[0].cpu().numpy()

        axes[i, 0].imshow(denormalize(img)); axes[i, 0].set_title("Ảnh gốc"); axes[i, 0].axis("off")
        axes[i, 1].imshow(PALETTE[mask.numpy().clip(0, NUM_CLASSES - 1)]); axes[i, 1].set_title("Ground Truth"); axes[i, 1].axis("off")
        axes[i, 2].imshow(PALETTE[pred.clip(0, NUM_CLASSES - 1)]); axes[i, 2].set_title("Dự đoán"); axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

print(f"✅ Model segmentation đã sẵn sàng tại: {CHECKPOINT_PATH}")
print("Bước tiếp theo: dùng notebook train_IL.ipynb để huấn luyện model điều khiển (Imitation Learning).")
